In [ ]:

# -- Cell 1 -- rclone + Drive.
# Settings -> Internet ON, Accelerator GPU T4 x2, RCLONE_DRIVE_TOKEN attached.
import os, subprocess

r = subprocess.run("curl -s https://rclone.org/install.sh | sudo bash", shell=True)
if r.returncode not in (0, 3):
    raise RuntimeError("rclone install failed (exit %d)" % r.returncode)

from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("RCLONE_DRIVE_TOKEN")
os.makedirs("/root/.config/rclone", exist_ok=True)
with open("/root/.config/rclone/rclone.conf", "w") as f:
    f.write("[drive]\ntype = drive\nscope = drive\ntoken = " + token + "\n")

REMOTE = "drive:Distillation"
out = subprocess.run("rclone lsf " + REMOTE, shell=True, capture_output=True, text=True)
print(out.stdout or out.stderr)
assert out.returncode == 0, "cannot see " + REMOTE


In [ ]:

# -- Cell 2 -- WHICH 16 BLOCKS? A size-matched slice experiment.
#
# THE QUESTION: are the last blocks doing work that only serves the pretraining
# objective? Downstream throws the MLM head away, so blocks that exist to predict
# masked tokens may contribute nothing to a pooled representation.
#
# Evidence so far says yes, weakly: depth 31 beat depth 32 in all four linear-probe
# measurements, and on THPep the 16-block prefix (0.8531) beat the full 32-block
# model (0.7764).
#
# THE DESIGN. Three models, all exactly 16 blocks and 168.8M parameters, differing
# only in WHICH 16:
#
#     prefix   blocks 0-15    already measured on THPep: MCC 0.8531
#     middle   blocks 8-23    new
#     suffix   blocks 16-31   new  <- the hypothesis under test
#
# Size is held constant, so any difference is about position, not capacity.
#
# WHY THE MIDDLE SLICE IS NOT OPTIONAL. Block 16 normally receives a residual
# stream built by blocks 0-15 (pooled norm ~324); fed raw embeddings instead
# (~99) it sees a distribution it never trained on. So a bad suffix score alone
# cannot separate "these blocks are useless" from "these blocks need their
# prefix". The middle slice has the same handicap. If middle survives and suffix
# does not, the handicap is not the explanation and the last blocks really are
# MLM-specialised; if both collapse, it is prefix-dependence and the experiment
# says nothing about the last blocks specifically.
subprocess.run('pip install -q -U "transformers>=5.0" peft lightning', shell=True, check=True)
subprocess.run("pip uninstall -y -q torchao", shell=True)

import torch, numpy as np, pandas as pd, glob, json, time
print("torch", torch.__version__, "| GPUs", torch.cuda.device_count())
NGPU = max(1, torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print("   cuda:%d %s %.0f GB" % (i, p.name, p.total_memory / 1e9))


In [ ]:

# -- Cell 3 -- their code and data, our code, the teacher.
WORK = "/kaggle/working"
CODE, REPO = WORK + "/distill", WORK + "/project/their_repo"
TEACH = WORK + "/models/peptideclm-2-mlm-large"
SMALL = WORK + "/models/peptideclm-2-mlm-small"
os.makedirs(REPO, exist_ok=True)

def rlsf(path):
    r = subprocess.run("rclone lsf " + path, shell=True, capture_output=True, text=True)
    return r.stdout.split() if r.returncode == 0 else []

def pull_model(name, local):
    if os.path.exists(local + "/model.safetensors"):
        return "already local"
    flat = "%s/models/%s" % (REMOTE, name)
    if any(f.startswith("model.safetensors") for f in rlsf(flat)):
        os.makedirs(local, exist_ok=True)
        subprocess.run("rclone copy %s %s -P" % (flat, local), shell=True, check=True)
        return "flat"
    snaps = "%s/models/models--aaronfeller--%s/snapshots" % (REMOTE, name)
    shas = [x.rstrip("/") for x in rlsf(snaps)]
    assert shas, "%s not found. Tried  %s  and  %s" % (name, flat, snaps)
    os.makedirs(local, exist_ok=True)
    subprocess.run("rclone copy %s/%s %s -P" % (snaps, shas[0], local), shell=True, check=True)
    return "snapshot " + shas[0][:12]

for sub in ("data", "training"):
    if not os.path.isdir(REPO + "/" + sub):
        subprocess.run("rclone copy %s/their_repo/%s %s/%s --transfers 16 -P"
                       % (REMOTE, sub, REPO, sub), shell=True, check=True)
subprocess.run("rclone copy %s/distill %s --transfers 8 -P" % (REMOTE, CODE),
               shell=True, check=True)
print("teacher <- %s" % pull_model("peptideclm-2-mlm-large", TEACH))
print("small   <- %s" % pull_model("peptideclm-2-mlm-small", SMALL))

TRAIN_PY = REPO + "/training/02_classification_benchmarks_training_code/scripts/classification_finetuning_v2.py"
DATA_DIR = REPO + "/data"

# bench_control.py and friends resolve paths from a project root; mirror the local
# checkout's layout so they work unchanged.
if not os.path.exists(WORK + "/their_repo"):
    os.symlink(REPO, WORK + "/their_repo")
for name, src in (("peptideclm-2-mlm-large", TEACH), ("peptideclm-2-mlm-small", SMALL)):
    d = "%s/models/models--aaronfeller--%s/snapshots/local" % (WORK, name)
    if not os.path.exists(d):
        os.makedirs(os.path.dirname(d), exist_ok=True)
        os.symlink(src, d)
assert os.path.exists(TRAIN_PY) and os.path.exists(CODE + "/export_truncated.py")
print("ok -- layout mirrored")


In [ ]:

# -- Cell 4 -- THPep split (their repo does not ship one) + controls.
r = subprocess.run(["python", "bench_control.py"], cwd=CODE,
                   capture_output=True, text=True)
print(r.stdout[-2000:])
if r.returncode != 0:
    print("----- STDERR -----"); print(r.stderr[-2000:])
assert r.returncode == 0, "control failed"


In [ ]:

# -- Cell 5 -- export the three slices.
#
# export_truncated.py verifies each two ways: exported block j must be
# bit-identical to source block keep[j] (pure tensor comparison, no forward pass,
# so rotary state cannot confound it), and the loaded model must reproduce the
# original with the same blocks bypassed.
EXPORT = WORK + "/compressed"
os.makedirs(EXPORT, exist_ok=True)

SLICES = {"prefix16": "0-15", "mid16": "8-23", "suffix16": "16-31"}
ARMS = {}
for name, keep in SLICES.items():
    out = "%s/peptideclm-2-mlm-%s" % (EXPORT, name)
    if not os.path.exists(out + "/model.safetensors"):
        r = subprocess.run(["python", "export_truncated.py", "--out", out,
                            "--keep", keep], cwd=CODE, capture_output=True, text=True)
        print("== %s (blocks %s) ==" % (name, keep)); print(r.stdout[-500:])
        if r.returncode != 0:
            print(r.stderr[-1200:]); continue
    ARMS[name] = out

# Identical parameter counts are the control. If these differ, the comparison is
# about capacity rather than position and the experiment is void.
sizes = {k: os.path.getsize(v + "/model.safetensors") for k, v in ARMS.items()}
print("\nsizes:", {k: "%.1f MB" % (v / 1e6) for k, v in sizes.items()})
assert len(set(sizes.values())) == 1, "slices differ in size -- not a fair comparison"


In [ ]:

# -- Cell 6 -- run their LoRA script on THPep, unmodified.
#
# THPep because it is the cheap one: 487 train / 122 test, and a 16-block arm took
# 18.7 min in the previous run. Three arms, two GPUs, well under an hour.
#
# THPep has no val file so their script takes the 5-fold CV branch: five training
# runs per job, each predicting the full test set, ensembled by mean logit below.
#
# --gpu_index must be passed: their Trainer does devices=[int(args.gpu_index)] on
# the raw argument, whose default is None.
OUT = WORK + "/results/slice_probe"
os.makedirs(OUT, exist_ok=True)
SEED = 101

todo = [a for a in ARMS
        if not glob.glob("%s/%s/seed_%d/*_results.csv" % (OUT, a, SEED))]
print("%d arms, %d to run" % (len(ARMS), len(todo)))

running, free, t0 = [], list(range(NGPU)), time.time()
while todo or running:
    while todo and free:
        arm = todo.pop(0); gpu = free.pop(0)
        d = "%s/%s/seed_%d" % (OUT, arm, SEED)
        os.makedirs(d, exist_ok=True)
        cmd = ["python", TRAIN_PY, "--dataset", "THPep", "--gpu", "0",
               "--gpu_index", "0", "--model_name", ARMS[arm],
               "--batch_size", "32", "--seed", str(SEED),
               "--data_dir", DATA_DIR, "--save_path", d,
               "--log_dir", "/tmp/logs/%s" % arm]
        p = subprocess.Popen(cmd, cwd=os.path.dirname(TRAIN_PY),
                             stdout=open(d + "/train.log", "w"),
                             stderr=subprocess.STDOUT,
                             env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)))
        running.append((arm, gpu, p, d))
        print("[%5.1f min] launch %-10s gpu%d" % ((time.time()-t0)/60, arm, gpu))
    time.sleep(20)
    for job in list(running):
        arm, gpu, p, d = job
        if p.poll() is None:
            continue
        running.remove(job); free.append(gpu)
        ok = p.returncode == 0 and glob.glob(d + "/*_results.csv")
        print("[%5.1f min] %-10s -> %s" % ((time.time()-t0)/60, arm,
                                           "ok" if ok else "FAILED rc=%s" % p.returncode))
        if not ok:
            print("".join(open(d + "/train.log").readlines()[-15:]))
print("")
print("done in %.1f min" % ((time.time() - t0) / 60))


In [ ]:

# -- Cell 7 -- read the slices.
from sklearn.metrics import matthews_corrcoef, roc_auc_score

rows = []
for f in sorted(glob.glob(OUT + "/*/seed_*/*_results.csv")):
    arm = os.path.basename(os.path.dirname(os.path.dirname(f)))
    d = pd.read_csv(f)
    if "fold" in d.columns and d.fold.nunique() > 1:
        d["i"] = d.groupby("fold").cumcount(); g = d.groupby("i")
        y, p = g.true_label.first().values, g.predicted_label.mean().values
    else:
        y, p = d.true_label.values, d.predicted_label.values
    rows.append(dict(slice=arm, blocks=SLICES.get(arm, "?"),
                     mcc=matthews_corrcoef(y, (p > 0).astype(int)),
                     auc=roc_auc_score(y, p), n=len(y)))
res = pd.DataFrame(rows)
print(res.to_string(index=False))

print("\nreference points on THPep (same 5-fold protocol, previous run):")
print("   full337M (32 blocks)   MCC 0.7764")
print("   warmstart32M (14 blk)  MCC 0.8431")
print("   bag-of-tokens control  MCC 0.6854")
print("   their published mlm-large            0.7557")
print("\nHOW TO READ THIS. THPep's test set is 122 molecules, so the bootstrap CI")
print("is roughly +-0.15 -- only a LARGE effect is readable here, which is exactly")
print("what this screen is for. Small orderings between slices mean nothing.")
print("   suffix collapses, middle survives -> last blocks are MLM-specialised,")
print("                                        drop them from later experiments")
print("   both collapse                     -> prefix-dependence, inconclusive")
print("                                        about the last blocks")
print("   suffix holds up                   -> the stack is far more modular than")
print("                                        expected; worth a second benchmark")

res.to_csv(OUT + "/slice_probe_metrics.csv", index=False)
subprocess.run("rclone copy %s %s/results/slice_probe --drive-chunk-size 64M -P"
               % (OUT, REMOTE), shell=True, check=True)
print("\nuploaded -> %s/results/slice_probe" % REMOTE)
